# 4 Smart Charging Using Reinforcement Learning

The previous sections studied *where* and *when* ride-hailing demand occurs. This section turns to the operational side of an electric fleet: given that a vehicle must be ready for its shift, *how should it charge*? We leave the Chicago trip data behind and study a single electric taxi that charges at home. The driver arrives at 2 p.m. and leaves at 4 p.m., so there is a two-hour window in which a charging agent sets the charging power every 15 minutes, giving eight sequential decisions. The energy the vehicle will need for the coming day is uncertain and is revealed only at departure, drawn from a normal distribution. Charging cost grows exponentially with power, and running out of energy is heavily penalised.

This makes the problem a sequential decision under uncertainty, which we formalise as a Markov decision process and solve with reinforcement learning. The agent has to balance two opposing forces: charge enough to cover an uncertain demand (safety), while spreading the load and exploiting cheaper time slots to keep cost down (economy).

The questions guiding this section are:

1. How is home charging formalised as a Markov decision process (states, actions, reward)?
2. Which charging policy minimises cost while avoiding energy shortfall under uncertain demand?
3. How close to the provable optimum does a learned DQN policy get, and does it beat naive charging strategies?
4. How does the policy react to the electricity price structure and to the level of demand uncertainty?

The section is organised as follows:

- *4.1 Problem Formalisation (MDP)*
- *4.2 Environment Implementation*
- *4.3 Design Choices and Parameters*
- *4.4 Reinforcement Learning Solution (DQN)*
- *4.5 Results: Policy and Evaluation against Baselines*
- *4.6 Sensitivity and Discussion*

The charging demand is modelled synthetically and independently of the trip data; where useful we anchor its parameters to realistic daily driving so the setup stays grounded rather than arbitrary.

## 4.1 Problem Formalisation (MDP)

Before writing any code we write the charging problem out as a **Markov Decision Process (MDP)**.
Think of an MDP as a precise recipe that answers four questions:

| Question | MDP component |
|---|---|
| *Where am I?* | **State** |
| *What can I do?* | **Action** |
| *What happens next?* | **Transition** |
| *How good was that?* | **Reward** |

Once those four things are defined, any RL algorithm (including DQN) can be plugged in.

---

### State  `s_t = (t, SoC_t)`

The state is everything the agent needs to make a good decision **right now**, without looking at history.
We use two numbers:

| Symbol | Meaning | Range |
|---|---|---|
| `t` | Which 15-minute slot we are in | 0 … 7  (14:00 → 15:45) |
| `SoC_t` | Battery charge right now (kWh) | 0 … B |

Why does `t` belong in the state? Because electricity is cheaper at some slots than others —
*when* we are changes what the best action is.

**Markov property:** given `(t, SoC_t)`, the future is independent of how we got here.
History is irrelevant because the state already captures everything that matters.

---

### Action  `a_t ∈ {0, 1, 2, 3}`

To keep the problem tractable for DQN (which needs discrete actions) we limit charging to four levels:

| Action index | Power (kW) | Energy added per 15-min slot (kWh) |
|---|---|---|
| 0 | 0 kW  | 0.00 |
| 1 | 3 kW  | 0.75 |
| 2 | 11 kW | 2.75 |
| 3 | 22 kW | 5.50 |

---

### Transition  (how the state changes)

Charging is **deterministic** — pushing 11 kW for 15 minutes always adds 2.75 kWh:

```
SoC_{t+1} = min(B, SoC_t + power(a_t) × 0.25)
```

The `min(B, ...)` cap prevents overcharging.

The **only randomness** is the daily energy demand `D ~ N(μ, σ)`, drawn once at departure (after slot 7).
The agent cannot see `D` during charging — it must plan conservatively.

---

### Reward  `r_t`

At each slot the agent pays an electricity cost that grows **exponentially** with the action index
(high-power charging is disproportionately expensive per unit of energy):

```
r_t = −α_t × exp(a_t)
```

where `α_t` is the electricity price coefficient at slot `t` (higher at peak hours),
and `a_t ∈ {0,1,2,3}` is the action index.

This gives cost ratios: off=×1, low=×2.7, medium=×7.4, high=×20 — a steep incentive to avoid max power.

At the **final step** an additional terminal reward fires:

```
terminal = −P    if SoC_final < D   (shortfall: vehicle cannot complete its shift)
terminal =  0    otherwise
```

`P` is calibrated to dominate any plausible charging cost, so the agent never trades safety for savings.

---

### Horizon and discount

| Parameter | Value | Reason |
|---|---|---|
| Horizon | 8 steps | Fixed 2-hour window, 15-min slots |
| Discount γ | 1.0 | All steps within one session matter equally |

---

### Assumptions

- Vehicle arrives at 14:00 with a fixed initial charge `SoC_init`.
- No driving between 14:00 and 16:00 (charging only).
- Demand `D` is revealed only after the last charging decision (at 16:00).
- The grid can always supply the requested power.

---

### MDP at a glance

| Component | Definition |
|---|---|
| State `s_t` | `(t, SoC_t)` — slot index and battery level (kWh) |
| Action `a_t` | Index in {0,1,2,3} → power in {0, 3, 11, 22} kW |
| Transition | `SoC_{t+1} = min(B, SoC_t + power(a_t) × 0.25)` |
| Reward | `−α_t × exp(a_t)` per slot; `−P` if `SoC_final < D` |
| Horizon | 8 steps (finite episode) |
| Discount γ | 1.0 |


## 4.2 Environment Implementation

Now we translate the MDP definition from 4.1 into Python code.

The environment is a **simulation of the world** — it knows nothing about strategy.
It just enforces the rules: apply an action, update the battery, return a reward.

We follow the standard `gym`-style interface with two methods:
- `reset()` — start a new episode (a fresh 2-hour charging session)
- `step(action)` — apply one charging decision, advance the clock, return what happened

This interface means the **same** environment can be used by the DQN agent,
the baselines, and the dynamic-programming solver — making comparisons fair.


In [ ]:
import numpy as np

class ChargingEnv:
    """
    Home EV charging environment (Gym-style interface).

    The agent controls charging power every 15 minutes over a 2-hour window
    (8 decision steps: 14:00 → 15:45).  At departure (after step 7) the
    daily energy demand D ~ N(mu, sigma) is drawn and a shortfall penalty
    applied if the battery cannot cover it.

    Interface
    ---------
    reset()        -> state (tuple)
    step(action)   -> (next_state, reward, done, info)
    """

    # Charging power levels (kW) for each discrete action index
    ACTION_POWERS = [0, 3, 11, 22]
    N_ACTIONS     = len(ACTION_POWERS)
    SLOT_HOURS    = 0.25    # 15 minutes = 0.25 hours

    def __init__(self, params: dict, seed: int = 42):
        """
        Parameters
        ----------
        params : dict with keys:
            mu, sigma  – demand distribution (kWh)
            B          – battery capacity (kWh)
            soc_init   – initial SoC at 14:00 (kWh)
            P          – shortfall penalty
            alpha      – list of 8 price coefficients (one per slot)
        seed : int
            Random seed for reproducible demand draws.
        """
        self.mu       = params['mu']
        self.sigma    = params['sigma']
        self.B        = params['B']
        self.soc_init = params['soc_init']
        self.P        = params['P']
        self.alpha    = np.array(params['alpha'])
        self.n_slots  = len(self.alpha)

        # Seedable RNG — same seed → same demand sequence across runs
        self.rng = np.random.default_rng(seed)

        self.reset()

    # ── reset: start a new charging session ──────────────────────────────────
    def reset(self):
        """Reset to 14:00 with initial SoC. Returns the starting state."""
        self.t          = 0
        self.soc        = float(self.soc_init)
        self.total_cost = 0.0
        self.trajectory = []
        return self._state()

    # ── step: apply one 15-minute charging decision ───────────────────────────
    def step(self, action: int):
        """
        Apply action for the current slot.

        Parameters
        ----------
        action : int  (0=off, 1=low, 2=medium, 3=high)

        Returns
        -------
        next_state : tuple (t+1, soc_{t+1})
        reward     : float  (negative cost; includes terminal penalty if done)
        done       : bool
        info       : dict   (diagnostic log for 4.5 analysis)
        """
        assert 0 <= action < self.N_ACTIONS, f"Invalid action {action}"

        power = self.ACTION_POWERS[action]   # kW

        # 1. Electricity cost — exponential in action index (not raw kW)
        #    This gives cost ratios: 0→×1, 1→×2.7, 2→×7.4, 3→×20
        slot_cost = self.alpha[self.t] * np.exp(action)
        reward    = -slot_cost

        # 2. Update battery (deterministic transition)
        energy_added = power * self.SLOT_HOURS        # kWh
        prev_soc     = self.soc
        self.soc     = min(self.B, self.soc + energy_added)

        # 3. Advance clock
        prev_t  = self.t
        self.t += 1
        self.total_cost += slot_cost

        # 4. Terminal step: draw demand and check for shortfall
        done      = (self.t == self.n_slots)
        demand    = None
        shortfall = False
        if done:
            demand    = float(self.rng.normal(self.mu, self.sigma))
            shortfall = (self.soc < demand)
            if shortfall:
                reward -= self.P

        # 5. Log this step (used in section 4.5 for trajectory analysis)
        info = {
            'slot'       : prev_t,
            'time_label' : f"{14 + prev_t // 4}:{(prev_t % 4) * 15:02d}",
            'action'     : action,
            'power_kw'   : power,
            'soc_before' : prev_soc,
            'soc_after'  : self.soc,
            'slot_cost'  : slot_cost,
            'demand'     : demand,
            'shortfall'  : shortfall,
        }
        self.trajectory.append(info)

        return self._state(), reward, done, info

    def _state(self):
        """Return current state as (t, SoC) with SoC rounded to 1 decimal."""
        return (self.t, round(self.soc, 1))

    def render_trajectory(self):
        """Print a human-readable episode summary."""
        header = f"{'Time':<8}{'Action':<8}{'kW':<6}{'SoC before':>12}{'SoC after':>11}{'Slot cost':>11}"
        print(header)
        print('-' * len(header))
        for s in self.trajectory:
            print(f"{s['time_label']:<8}{s['action']:<8}{s['power_kw']:<6}"
                  f"{s['soc_before']:>12.2f}{s['soc_after']:>11.2f}{s['slot_cost']:>11.4f}")
        last = self.trajectory[-1]
        print('-' * len(header))
        print(f"Final SoC : {last['soc_after']:.2f} kWh")
        print(f"Demand    : {last['demand']:.2f} kWh")
        print(f"Shortfall : {last['shortfall']}")
        print(f"Total cost: {self.total_cost:.4f}")

print("ChargingEnv class defined.")


### Sanity check

Before any learning, we run one **dummy policy** (always charge at medium = action 2, i.e. 11 kW)
and print the full trajectory. This confirms:
- The battery increments correctly each slot
- The battery cap `B` works (SoC never exceeds B)
- The slot cost uses `α_t × exp(action)` — higher at peak slots
- The shortfall penalty fires when `SoC_final < D`


In [ ]:
# We use placeholder params here; the full justified set is defined in 4.3.
_params_preview = {
    'mu'      : 30.0,
    'sigma'   : 5.0,
    'B'       : 40.0,
    'soc_init': 5.0,
    'P'       : 500.0,
    'alpha'   : [0.05, 0.05, 0.08, 0.12, 0.18, 0.18, 0.12, 0.08],
}

env_check = ChargingEnv(_params_preview, seed=0)
env_check.reset()

done = False
total_reward = 0.0
while not done:
    _, reward, done, _ = env_check.step(action=2)   # always medium
    total_reward += reward

env_check.render_trajectory()
print(f"\nTotal episode reward : {total_reward:.4f}")
print("\nNote: shortfall occurred because 8 × 2.75 kWh = 22 kWh added, 5 + 22 = 27 < demand.")
print("The agent needs a smarter policy — that is what DQN learns in 4.4.")


## 4.3 Design Choices and Parameters

Every parameter is **justified** here, not just stated.
A poor choice can make the problem trivial (always enough charge regardless of policy)
or impossible (cannot reach safe charge in the 2-hour window).


In [ ]:
# ── Demand distribution ───────────────────────────────────────────────────────
# Chicago taxi data (Task 1) shows average daily distance ~150 km.
# A typical EV consumes ~0.20 kWh/km → expected daily need ≈ 30 kWh.
# σ = 5 kWh (≈17% coefficient of variation) reflects realistic day-to-day variation.
MU    = 30.0   # kWh — expected daily energy need
SIGMA =  5.0   # kWh — standard deviation

# ── Battery capacity ───────────────────────────────────────────────────────────
# B = 40 kWh gives a ~10 kWh safety margin above μ = 30 kWh.
# Feasibility: from SoC_init=5 kWh, max charging (22 kW × 8 × 0.25h = 44 kWh)
# reaches min(40, 5+44) = 40 kWh, comfortably above μ+2σ = 40 kWh.
B        = 40.0   # kWh — battery capacity
SOC_INIT =  5.0   # kWh — charge at 14:00 (driver arrives nearly depleted after the night)

# ── Action set ────────────────────────────────────────────────────────────────
# Four levels give the agent meaningful choices without exploding the action space.
# {0, 3, 11, 22} kW covers off / trickle / moderate / fast charging.
ACTION_POWERS = [0, 3, 11, 22]   # kW

# ── Shortfall penalty ─────────────────────────────────────────────────────────
# Maximum plausible charging cost = 8 slots × max alpha × exp(3)
# = 8 × 0.18 × 20.09 ≈ 28.9
# Setting P = 500 is >> 28.9, so the agent always prefers safety over any savings.
P = 500.0

# ── Electricity price profile (time-of-use) ───────────────────────────────────
# A flat profile would make slot timing irrelevant — nothing to learn about *when* to charge.
# We use a realistic afternoon/evening-peak structure (typical German day-ahead prices):
#   14:00–14:15 (slots 0–1): off-peak, cheap   → agent should charge here
#   14:30–14:45 (slots 2–3): price rising
#   15:00–15:15 (slots 4–5): peak, expensive   → agent should avoid charging here
#   15:30–15:45 (slots 6–7): price falling
ALPHA = [0.05, 0.05, 0.08, 0.12, 0.18, 0.18, 0.12, 0.08]
TIMES = ['14:00','14:15','14:30','14:45','15:00','15:15','15:30','15:45']

# ── SoC discretisation (for dynamic programming in 4.4) ───────────────────────
SOC_STEP = 0.5   # kWh per step → 81 levels (0.0, 0.5, ..., 40.0)

# ── Bundle into a single params dict (used by ChargingEnv and all agents) ─────
PARAMS = {
    'mu'      : MU,
    'sigma'   : SIGMA,
    'B'       : B,
    'soc_init': SOC_INIT,
    'P'       : P,
    'alpha'   : ALPHA,
}

# ── Print parameter summary ───────────────────────────────────────────────────
print("=== Task 4 Environment Parameters ===")
print(f"  Demand distribution : N(μ={MU}, σ={SIGMA}) kWh")
print(f"  Battery capacity    : B = {B} kWh")
print(f"  Initial SoC         : {SOC_INIT} kWh")
print(f"  Shortfall penalty   : P = {P}")
print(f"  Action powers       : {ACTION_POWERS} kW")
print(f"  Max energy addable  : {max(ACTION_POWERS)*0.25*8:.1f} kWh (22 kW × 8 slots)")
print(f"  State space size    : 8 slots × {int(B/SOC_STEP)+1} SoC levels = {8*(int(B/SOC_STEP)+1)} states")
print()
print("  Electricity price profile:")
for t, a in zip(TIMES, ALPHA):
    bar = '█' * int(a * 100)
    print(f"    {t}  α={a:.2f}  {bar}")
print()
print("  Cost ratios by action (α_t × exp(action_index)):")
import numpy as np
for i, kw in enumerate(ACTION_POWERS):
    print(f"    action {i} ({kw:>2} kW): multiplier = exp({i}) = {np.exp(i):.3f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── Left: SoC trajectory under max charging ────────────────────────────────
soc_traj = [SOC_INIT]
for _ in range(8):
    soc_traj.append(min(B, soc_traj[-1] + 22 * 0.25))

axes[0].step(range(9), soc_traj, where='post', color='#185FA5', linewidth=2.5, label='Max power trajectory')
axes[0].axhline(MU,          color='#D85A30', linestyle='--', linewidth=1.5, label=f'μ = {MU} kWh')
axes[0].axhline(MU + 2*SIGMA, color='#D85A30', linestyle=':',  linewidth=1.5, label=f'μ+2σ = {MU+2*SIGMA} kWh')
axes[0].axhline(B,            color='#3B6D11', linestyle='--', linewidth=1.5, label=f'B = {B} kWh')
axes[0].fill_between(range(9), MU, MU+2*SIGMA, alpha=0.08, color='#D85A30')
axes[0].set_xlabel('Slot')
axes[0].set_ylabel('SoC (kWh)')
axes[0].set_title('Feasibility check')
axes[0].set_xticks(range(9))
axes[0].set_xticklabels(TIMES + ['16:00'], rotation=30, fontsize=8)
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

# ── Middle: demand distribution ────────────────────────────────────────────
rng = np.random.default_rng(0)
samples = rng.normal(MU, SIGMA, 20000)
axes[1].hist(samples, bins=60, color='#5DCAA5', edgecolor='none', alpha=0.85, density=True)
axes[1].axvline(MU,           color='#D85A30', linestyle='--', linewidth=1.5, label=f'μ={MU}')
axes[1].axvline(MU + 2*SIGMA, color='#D85A30', linestyle=':',  linewidth=1.5, label=f'μ+2σ={MU+2*SIGMA}')
axes[1].axvline(B,            color='#185FA5', linestyle='--', linewidth=1.5, label=f'B={B}')
axes[1].set_xlabel('Daily demand (kWh)')
axes[1].set_ylabel('Density')
axes[1].set_title('Demand distribution N(30, 5)')
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

# ── Right: electricity price profile ──────────────────────────────────────
bar_colors = ['#5DCAA5' if a <= 0.08 else '#EF9F27' if a <= 0.12 else '#D85A30' for a in ALPHA]
axes[2].bar(range(8), ALPHA, color=bar_colors, edgecolor='none', alpha=0.85)
axes[2].set_xlabel('Slot')
axes[2].set_ylabel('Price coefficient α_t')
axes[2].set_title('Time-of-use electricity price')
axes[2].set_xticks(range(8))
axes[2].set_xticklabels(TIMES, rotation=30, fontsize=8)
axes[2].grid(True, alpha=0.3, axis='y')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#5DCAA5', label='Cheap'),
                   Patch(facecolor='#EF9F27', label='Mid'),
                   Patch(facecolor='#D85A30', label='Peak')]
axes[2].legend(handles=legend_elements, fontsize=7)

plt.suptitle('Section 4.3 — Parameter Justification', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_43_parameters.png', dpi=120, bbox_inches='tight')
plt.show()
print("Feasibility: max charging reaches 40.0 kWh = B = μ+2σ — the 95th-percentile demand is just coverable.")
print(f"Penalty P={P} >> max total cost ≈ {8*max(ALPHA)*np.exp(3):.1f} — safety always dominates.")


## 4.4 Reinforcement Learning Solution (DQN)

Solve the MDP with a Deep Q-Network, the method the course covers for discrete-action control. This subsection has to deliver:

- A Q-network mapping the state to one Q-value per action, trained with epsilon-greedy exploration, an experience replay buffer and a target network. Report the architecture and all hyperparameters.
- Training over many episodes, each a fresh charging session with a newly drawn demand, plus a convergence plot (episode reward and loss) that demonstrates the agent actually learns rather than merely runs.
- A provable reference: because the state space is small, also compute the exact optimal policy by dynamic programming (value iteration over the discretised MDP). This optimum is the yardstick the DQN is measured against in 4.5 and is what lifts the evaluation from descriptive to rigorous.

Show the DQN learning curve approaching the dynamic-programming optimum, so convergence is quantified and not just asserted. A tabular Q-learning agent may be added as a lightweight second learner, but the DP optimum is the reference that matters.

## 4.5 Results: Policy and Evaluation against Baselines

Demonstrate that the learned policy is both safe and economical, and benchmark it properly. This subsection has to provide:

- A visualisation of the learned policy: the chosen action as a function of the state `(t, SoC)`, and the charging schedule of a representative episode, so the behaviour is readable rather than a black box.
- An evaluation over many independent test episodes on two metrics that capture the trade-off: mean recharging cost and shortfall rate (how often the vehicle runs out of energy). Reporting only one of the two is not enough.
- A benchmark table comparing the DQN against the dynamic-programming optimum and against naive baselines: constant charging, greedy charge-as-fast-as-possible, and a cheapest-slots heuristic. The DQN should sit close to the optimum and clearly dominate the naive strategies.
- Distributions, not only averages: show the spread of cost and the rare shortfall events, since a fleet operator cares about the worst cases, not just the mean.

This benchmarking layer is what turns "we built an RL agent" into "our agent is provably good", and it mirrors the evaluation rigour applied to the predictive models in Section 3.

## 4.6 Sensitivity and Discussion

Show that the result is robust and translate it into advice, with a few targeted experiments rather than an exhaustive grid. This subsection has to cover:

- Vary the electricity price profile from flat to steeply peaked and show how charging concentrates into the cheap slots as the profile steepens.
- Vary the demand uncertainty `sigma` and show that the policy keeps a larger safety buffer as uncertainty grows, making the safety-versus-cost trade-off explicit.
- Briefly vary the penalty `P` and the maximum power to confirm the policy still reacts sensibly.
- Interpret in plain terms what the agent has learned, then connect it to the business question of Section 5: when home charging is sufficient, and what this implies for the choice between private and public charging infrastructure.

Close by stating the limitations honestly (a single vehicle, a synthetic demand, a stylised price profile), so the scope of the conclusion is clear and not overstated.